In [3]:
# Импорты

import random
from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import os
import sys

d:\DataScience\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Пути

ROOT = Path.cwd().parent

DATASET_PATH = ROOT / "data" / "processed"
MODELS_PATH = ROOT / "models"

# Создаем директорию для сохранения результатов
MODELS_PATH.mkdir(exist_ok=True)

In [5]:
from pathlib import Path
sys.path.append(os.path.abspath(".."))

In [6]:
from src.data import (
    collect_dataset,
    AnimeDataset,
    TripletDataset,
)
from src.model import EmbeddingNet

from src.train import (
    train_triplet_epoch,
    validate_triplet,
)

In [7]:
# Параметры обучения

IMAGE_SIZE = 224       # Размер входного изображения
EMBEDDING_SIZE = 512   # Размерность выходного эмбеддинга

BATCH_SIZE = 32        # Размер батча
EPOCHS = 20            # Количество эпох обучения
LR = 1e-4              # Скорость обучения
MARGIN = 0.3           # Margin для Triplet Loss

SEED = 42              # Начальное значение генератора случайных чисел

In [8]:
# Определяем устройства

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [9]:
# Сбор информации об изображениях датасета

images = []

for anime_dir in sorted(DATASET_PATH.iterdir()):
    # Пропускаем файлы, оставляя только директории с классами
    if not anime_dir.is_dir():
        continue

    label = anime_dir.name

    for image_path in anime_dir.glob("*"):
        # Добавляем только изображения поддерживаемых форматов
        if image_path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}:
            images.append((image_path, label))

print(len(images))

24009


In [10]:
# Разделение датасета на обучающую, валидационную и тестовую выборки

# Получаем список меток для стратифицированного разделения
labels = [label for _, label in images]

# Разделяем датасет на обучающую (70%) и временную (30%) выборки
train_images, temp_images = train_test_split(
    images,
    test_size=0.30,
    stratify=labels,
    random_state=SEED
)

# Получаем метки временной выборки
temp_labels = [label for _, label in temp_images]

# Разделяем временную выборку на валидационную (15%) и тестовую (15%)
val_images, test_images = train_test_split(
    temp_images,
    test_size=0.50,
    stratify=temp_labels,
    random_state=SEED
)

print(len(train_images))
print(len(val_images))
print(len(test_images))

16806
3601
3602


In [11]:
# Преобразования изображений

# Преобразования для обучающей выборки
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Преобразования для валидационной и тестовой выборок
test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [12]:
# Создание датасетов

train_dataset = TripletDataset(
    train_images,
    train_transform
)

val_dataset = TripletDataset(
    val_images,
    test_transform
)

test_dataset = TripletDataset(
    test_images,
    test_transform
)

# Создание DataLoader'ов

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [13]:
# Создание модели

# Инициализируем модель и переносим ее на выбранное устройство
model = EmbeddingNet(
    embedding_size=EMBEDDING_SIZE
).to(device)

print(model)

EmbeddingNet(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
    

In [14]:
# Настройка обучения

# Функция потерь для обучения эмбеддингов
criterion = nn.TripletMarginLoss(
    margin=MARGIN,
    p=2
)

# Оптимизатор модели
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

# Планировщик изменения скорости обучения
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

In [15]:
# Проверка формы батча

# Получаем первый батч из обучающего DataLoader
anchor, positive, negative = next(iter(train_loader))

print(anchor.shape)
print(positive.shape)
print(negative.shape)

torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])


In [18]:
# Обучение модели

best_loss = float("inf")
patience = 5
counter = 0

# Логирование обучения в TensorBoard
writer = SummaryWriter("../logs/resnet50_triplet")

for epoch in range(EPOCHS):
    # Обучение и валидация
    train_loss = train_triplet_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss = validate_triplet(
        model,
        val_loader,
        criterion,
        device
    )

    # Обновляем планировщик скорости обучения
    scheduler.step()

    # Сохраняем метрики в TensorBoard
    writer.add_scalar("Loss/Train", train_loss, epoch)
    writer.add_scalar("Loss/Validation", val_loss, epoch)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train: {train_loss:.4f} | "
        f"Validation: {val_loss:.4f}"
    )

    # Сохраняем модель при улучшении качества
    if val_loss < best_loss:
        best_loss = val_loss
        counter = 0

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "embedding_size": EMBEDDING_SIZE,
                "image_size": IMAGE_SIZE,
                "epoch": epoch
            },
            MODELS_PATH / "best_triplet_resnet50.pth"
        )

    else:
        counter += 1

        # Останавливаем обучение при отсутствии улучшений
        if counter >= patience:
            print("Early stopping.")
            break

writer.close()

100%|██████████| 526/526 [09:51<00:00,  1.12s/it]


Epoch 1/20 | Train: 0.1911 | Validation: 0.1517


 68%|██████▊   | 357/526 [36:29<17:16,  6.13s/it]


KeyboardInterrupt: 